# Olympic AI Sinh viên 2026 — Tác vụ 1: R-ViHSD
### Đội **FPTU_Promt_Engineer** · tài khoản `foa25`

**Điểm Private cuối cùng: 0.720**

Hệ thống NLP đa nhiệm cho bình luận mạng xã hội tiếng Việt: dự đoán đồng thời
mức độ hate speech (3 lớp) và loại nhiễu văn bản (7 lớp).

```
Score = 0.85 · MacroF1(hate) + 0.15 · MacroF1(noise)
```

**Tuân thủ quy định:** mô hình học máy thật (ViSoBERT ~98M tham số fine-tune),
seed cố định toàn bộ, không dùng dữ liệu ngoài, **không dùng cột `id`** làm đặc
trưng — `id` chỉ để ghép dự đoán vào đúng dòng khi xuất CSV.

## 1. Thiết lập

In [1]:
import os, re, sys, json, time, random, unicodedata, collections, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

DATA  = "data"
SEED  = 42
MODEL = "5CD-AI/visobert-14gb-corpus"      # encoder pretrain trên mạng xã hội tiếng Việt
HATE_LABELS  = ["CLEAN", "OFFENSIVE", "HATE"]
NOISE_LABELS = ["ORIGINAL","NO_DIACRITICS","TEENCODE","CHAR_REPEAT","PUNCT_NOISE","OBFUSCATION","MIXED"]

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); os.environ["PYTHONHASHSEED"]=str(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

torch 2.6.0+cu124 | cuda True


## 2. Dữ liệu

In [2]:
tr = pd.read_csv(f"{DATA}/training_set.csv"); va = pd.read_csv(f"{DATA}/validation_set.csv")
df = pd.concat([tr, va], ignore_index=True)
df["text"] = df.text.fillna("").astype(str)
df = df.drop_duplicates(subset=["text","label","noise_type"]).reset_index(drop=True)
df["y_hate"]  = df.label.map({l:i for i,l in enumerate(HATE_LABELS)})
df["y_noise"] = df.noise_type.map({l:i for i,l in enumerate(NOISE_LABELS)})
print("train+val sau khử trùng lặp:", len(df))
print()
print("phân bố hate:");  print((df.label.value_counts(normalize=True)*100).round(2).to_string())
print()
print("phân bố noise:"); print((df.noise_type.value_counts(normalize=True)*100).round(2).to_string())

train+val sau khử trùng lặp: 51663

phân bố hate:
label
CLEAN        82.34
HATE         10.78
OFFENSIVE     6.88

phân bố noise:
noise_type
ORIGINAL         48.48
OBFUSCATION       8.74
CHAR_REPEAT       8.65
TEENCODE          8.56
MIXED             8.54
PUNCT_NOISE       8.52
NO_DIACRITICS     8.51


### 2.1 Phát hiện then chốt: tập train ghép cặp

Mỗi comment gốc xuất hiện **hai lần** — một bản `ORIGINAL` và một bản đã biến đổi.
Nếu chia fold ngẫu nhiên, hai bản của cùng một comment rơi vào train và validation
khác nhau → **rò rỉ**, CV cao giả khoảng **+0.04**.

Khoá chuỗi thông thường không ghép được `OBFUSCATION` (một ký tự bị thay bằng
`*`, `.` hoặc `_` nên không chuẩn hoá về được), nên phải dùng so khớp
nearest-neighbour theo char n-gram.

In [3]:
def strip_diacritics(s):
    s = unicodedata.normalize("NFD", str(s))
    return "".join(c for c in s if unicodedata.category(c)!="Mn").replace("đ","d").replace("Đ","D")

def match_form(s):
    s = strip_diacritics(s).lower()
    s = re.sub(r"[^a-z0-9]", "", s)         # bỏ cả dấu câu lẫn ký hiệu obfuscation
    return re.sub(r"(.)\1+", r"\1", s)      # gộp ký tự lặp

df["mf"] = df.text.map(match_form)
is_o = (df.noise_type=="ORIGINAL").values
oi, ni = np.where(is_o)[0], np.where(~is_o)[0]
vec = TfidfVectorizer(analyzer="char", ngram_range=(3,4), min_df=1, sublinear_tf=True)
O = normalize(vec.fit_transform(df.mf.values[oi])); N = normalize(vec.transform(df.mf.values[ni]))

grp = np.arange(len(df)); first = {}
for i in oi: grp[i] = first.setdefault(df.mf.iat[i], i)
bi = np.zeros(len(ni),int); bs = np.zeros(len(ni))
for a in range(0, len(ni), 2000):
    S = (N[a:a+2000] @ O.T).toarray(); bi[a:a+2000]=S.argmax(1); bs[a:a+2000]=S.max(1)
hit = bs >= 0.70
grp[ni[hit]] = grp[oi[bi[hit]]]
df["group"] = ["G%d"%g for g in grp]
print(f"ghép được {hit.sum()}/{len(ni)} mẫu nhiễu về comment gốc ({hit.mean()*100:.1f}%)")
print(pd.Series(hit, index=df.noise_type.values[ni]).groupby(level=0).mean().round(3).to_string())
print(f"\nsố nhóm comment gốc: {df.group.nunique()}")

ghép được 24471/26615 mẫu nhiễu về comment gốc (91.9%)
CHAR_REPEAT      0.987
MIXED            0.859
NO_DIACRITICS    0.986
OBFUSCATION      0.743
PUNCT_NOISE      0.984
TEENCODE         0.961

số nhóm comment gốc: 26113


In [4]:
strat = df.y_hate.astype(str) + "_" + df.y_noise.astype(str)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
df["fold"] = -1
for k,(_,te) in enumerate(sgkf.split(df, strat, groups=df.group)): df.loc[te,"fold"] = k
print("rò rỉ nhóm giữa các fold:", (df.groupby("group").fold.nunique()>1).sum())
print(pd.crosstab(df.fold, df.label).to_string())

rò rỉ nhóm giữa các fold: 0
label  CLEAN  HATE  OFFENSIVE
fold                         
0       8615  1121        745
1       8596  1095        694
2       8436  1152        690
3       8681  1085        758
4       8209  1118        668


### 2.2 Trần Bayes của nhãn nhiễu

BTC xác nhận `noise_type` ghi lại **thao tác biến đổi đã chạy**, không phải hình
thức bề mặt của kết quả. Với chuỗi không có chỗ cho phép biến đổi kích hoạt, văn
bản ra **y hệt** văn bản vào nhưng nhãn vẫn được ghi.

In [5]:
orig = {g: s.text.iloc[0] for g,s in df[df.noise_type=="ORIGINAL"].groupby("group")}
d = df[(df.noise_type!="ORIGINAL") & df.group.map(orig).notna()].copy()
d["noop"] = d.text.str.strip() == d.group.map(orig).str.strip()
t = d.groupby("noise_type").noop.agg(["mean","sum","count"]); t.columns=["ti_le_noop","so_noop","tong"]
print(t.sort_values("ti_le_noop", ascending=False).round(3).to_string())
print(f"\n=> {t.loc['TEENCODE','ti_le_noop']*100:.1f}% mẫu TEENCODE không thể phân biệt với ORIGINAL.")
print("   Trần lý thuyết của noise macro-F1 khoảng 0.78 — không phải lỗi mô hình.")

               ti_le_noop  so_noop  tong
noise_type                              
TEENCODE            0.767     3260  4252
OBFUSCATION         0.102      342  3355
NO_DIACRITICS       0.054      236  4331
MIXED               0.008       30  3791
CHAR_REPEAT         0.005       24  4410
PUNCT_NOISE         0.000        0  4332

=> 76.7% mẫu TEENCODE không thể phân biệt với ORIGINAL.
   Trần lý thuyết của noise macro-F1 khoảng 0.78 — không phải lỗi mô hình.


## 3. Mô hình: ViSoBERT đa nhiệm + FGM + consistency

In [6]:
class DS(Dataset):
    def __init__(self, texts, tok, maxlen, yh=None, yn=None, pair=None):
        self.t, self.tok, self.m, self.yh, self.yn, self.pair = list(texts), tok, maxlen, yh, yn, pair
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        e = self.tok(self.t[i], truncation=True, max_length=self.m)
        o = {"input_ids": e["input_ids"], "attention_mask": e["attention_mask"]}
        if self.yh is not None:
            o["yh"], o["yn"] = int(self.yh[i]), int(self.yn[i])
        if self.pair is not None:
            j = self.pair[i]; j = i if j < 0 else int(j)
            e2 = self.tok(self.t[j], truncation=True, max_length=self.m)
            o.update(p_input_ids=e2["input_ids"], p_attention_mask=e2["attention_mask"],
                     p_yh=int(self.yh[j]), p_yn=int(self.yn[j]), p_ok=float(self.pair[i] >= 0))
        return o

def _pad(b, k, pv):
    L = max(len(x[k]) for x in b); out = torch.full((len(b),L), pv, dtype=torch.long)
    for i,x in enumerate(b): out[i,:len(x[k])] = torch.tensor(x[k])
    return out

def collate(b, pad):
    o = {"input_ids": _pad(b,"input_ids",pad), "attention_mask": _pad(b,"attention_mask",0)}
    if "yh" in b[0]:
        o["yh"]=torch.tensor([x["yh"] for x in b]); o["yn"]=torch.tensor([x["yn"] for x in b])
    if "p_input_ids" in b[0]:
        o["p_input_ids"]=_pad(b,"p_input_ids",pad); o["p_attention_mask"]=_pad(b,"p_attention_mask",0)
        o["p_yh"]=torch.tensor([x["p_yh"] for x in b]); o["p_yn"]=torch.tensor([x["p_yn"] for x in b])
        o["p_ok"]=torch.tensor([x["p_ok"] for x in b])
    return o

class MTL(nn.Module):
    """Encoder chia sẻ, hai đầu ra. Nhận biết nhiễu vừa được chấm điểm (15%),
    vừa là tác vụ phụ ép encoder học biểu diễn bền — thí nghiệm cho thấy giảm
    trọng số nhiễu xuống 0.05 làm hate TỆ đi 0.008."""
    def __init__(self, name):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name)
        h = self.enc.config.hidden_size
        self.drop = nn.Dropout(0.1)
        self.hate, self.noise = nn.Linear(h*2,3), nn.Linear(h*2,7)
    def forward(self, ids, att):
        o = self.enc(input_ids=ids, attention_mask=att).last_hidden_state
        m = att.unsqueeze(-1).float()
        mean = (o*m).sum(1)/m.sum(1).clamp(min=1e-6)
        mx = o.masked_fill(m==0,-1e4).max(1).values
        z = self.drop(torch.cat([mean,mx],-1))
        return self.hate(z), self.noise(z)

class FGM:
    """Nhiễu đối kháng trên word embedding. Bài thi chấm chính khả năng bền
    trước biến dạng, nên đây là kỹ thuật khớp nhất — đo được +0.0121 OOF,
    dương trên cả 5/5 fold."""
    def __init__(self, model, eps=1.0, key="word_embeddings"):
        self.m, self.eps, self.key, self.bak = model, eps, key, {}
    def attack(self):
        for n_,p in self.m.named_parameters():
            if p.requires_grad and self.key in n_ and p.grad is not None:
                self.bak[n_] = p.data.clone()
                nrm = torch.norm(p.grad)
                if nrm != 0 and not torch.isnan(nrm): p.data.add_(self.eps*p.grad/nrm)
    def restore(self):
        for n_,p in self.m.named_parameters():
            if n_ in self.bak: p.data = self.bak[n_]
        self.bak = {}
print("đã định nghĩa mô hình")

đã định nghĩa mô hình


### 3.1 Consistency loss trên các cặp THẬT

Không sinh cặp tổng hợp. Tập train **đã sẵn** ~16.000 cặp `(gốc, bản nhiễu)` của
cùng một comment, tạo bởi chính phép biến đổi của BTC — độ trung thực cao hơn bất
kỳ augmentation nào ta tự chế (thí nghiệm cho thấy augment tự chế **âm 0.0032**).

Chỉ ràng buộc **đầu hate** phải đồng thuận. **Không** ràng buộc đầu noise: hai
view cùng một comment nhưng nhãn nhiễu **bắt buộc phải khác nhau**.

In [7]:
def run_fold(df, k, tok, te_texts, epochs=8, bs=64, lr=2e-5, head_lr=1e-3,
             maxlen=96, w_noise=0.30, fgm_eps=1.0, cons=0.0, tag=None, seed=SEED):
    set_seed(seed + k)
    dev = "cuda"
    trn, val = df[df.fold!=k], df[df.fold==k]
    pair = None
    if cons > 0:
        pos = {ix:i for i,ix in enumerate(trn.index)}
        pair = np.full(len(trn), -1, dtype=np.int64)
        for _, sub in trn.groupby("group"):
            if len(sub) < 2: continue
            ori = sub.index[sub.noise_type.values=="ORIGINAL"]
            for ix in sub.index:
                cand = [c for c in (ori if len(ori) else sub.index) if c != ix]
                if cand: pair[pos[ix]] = pos[cand[0]]
    cf = lambda b: collate(b, tok.pad_token_id)
    dl_tr = DataLoader(DS(list(trn.text), tok, maxlen, trn.y_hate.values, trn.y_noise.values, pair),
                       batch_size=bs, shuffle=True, collate_fn=cf, num_workers=4, drop_last=True)
    dl_va = DataLoader(DS(list(val.text), tok, maxlen), batch_size=256, collate_fn=cf, num_workers=4)
    dl_te = DataLoader(DS(te_texts, tok, maxlen), batch_size=256, collate_fn=cf, num_workers=4)

    model = MTL(MODEL).to(dev)
    wh = torch.tensor(len(trn)/(3*np.bincount(trn.y_hate,minlength=3)), dtype=torch.float, device=dev)
    wn = torch.tensor(len(trn)/(7*np.bincount(trn.y_noise,minlength=7)), dtype=torch.float, device=dev)
    head = [n_ for n_,_ in model.named_parameters() if n_.startswith(("hate","noise"))]
    opt = torch.optim.AdamW([
        {"params":[q for n_,q in model.named_parameters() if n_ not in head], "lr":lr},
        {"params":[q for n_,q in model.named_parameters() if n_ in head], "lr":head_lr}], weight_decay=0.01)
    steps = len(dl_tr)*epochs
    sch = get_cosine_schedule_with_warmup(opt, int(0.1*steps), steps)
    fgm = FGM(model, fgm_eps) if fgm_eps>0 else None

    def infer(dl):
        model.eval(); H,N_=[],[]
        with torch.no_grad(), torch.amp.autocast("cuda",dtype=torch.bfloat16):
            for b in dl:
                lh,ln = model(b["input_ids"].to(dev), b["attention_mask"].to(dev))
                H.append(lh.float().softmax(-1).cpu()); N_.append(ln.float().softmax(-1).cpu())
        return torch.cat(H).numpy(), torch.cat(N_).numpy()

    best = (-1,None,None)
    for ep in range(epochs):
        model.train(); t0=time.time(); tot=0
        for b in dl_tr:
            ids,att = b["input_ids"].to(dev), b["attention_mask"].to(dev)
            th,tn = b["yh"].to(dev), b["yn"].to(dev)
            def _loss():
                with torch.amp.autocast("cuda",dtype=torch.bfloat16):
                    lh,ln = model(ids,att)
                    L = (1-w_noise)*F.cross_entropy(lh,th,weight=wh) + w_noise*F.cross_entropy(ln,tn,weight=wn)
                    if cons>0:
                        lh2,ln2 = model(b["p_input_ids"].to(dev), b["p_attention_mask"].to(dev))
                        ok = b["p_ok"].to(dev)
                        L = 0.5*L + 0.5*((1-w_noise)*F.cross_entropy(lh2,b["p_yh"].to(dev),weight=wh)
                                         + w_noise*F.cross_entropy(ln2,b["p_yn"].to(dev),weight=wn))
                        q1,q2 = lh.float().log_softmax(-1), lh2.float().log_softmax(-1)
                        kl = 0.5*(F.kl_div(q1,q2,log_target=True,reduction="none").sum(-1)
                                  + F.kl_div(q2,q1,log_target=True,reduction="none").sum(-1))
                        L = L + cons*(kl*ok).sum()/ok.sum().clamp(min=1)
                    return L
            opt.zero_grad(set_to_none=True)
            loss=_loss(); loss.backward()
            if fgm is not None: fgm.attack(); _loss().backward(); fgm.restore()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step(); sch.step(); tot+=loss.item()
        ph,pn = infer(dl_va)
        fh = f1_score(val.y_hate, ph.argmax(1), average="macro", labels=range(3), zero_division=0)
        fn = f1_score(val.y_noise, pn.argmax(1), average="macro", labels=range(7), zero_division=0)
        s = 0.85*fh + 0.15*fn
        print(f"  fold{k} ep{ep+1} loss={tot/len(dl_tr):.4f} hate={fh:.4f} noise={fn:.4f} SCORE={s:.4f} ({time.time()-t0:.0f}s)", flush=True)
        if s > best[0]:
            best = (s,(ph,pn),infer(dl_te))
            if tag:
                os.makedirs(f"ckpt/{tag}", exist_ok=True)
                torch.save({"model":model.state_dict(),"fold":k,"epoch":ep+1,"score":s}, f"ckpt/{tag}/fold{k}.pt")
    del model; torch.cuda.empty_cache()
    return best, val.index.values
print("đã định nghĩa vòng huấn luyện")

đã định nghĩa vòng huấn luyện


## 4. Huấn luyện

Ba mô hình được huấn luyện trong vòng thi, mỗi mô hình 5 fold, tổng 15 checkpoint:

| tag | cấu hình | OOF |
|---|---|---|
| `visobert_clean` | ViSoBERT đa nhiệm, `w_noise=0.30` | 0.7018 |
| `visobert_fgm`   | + FGM ε=1.0 | 0.7139 |
| `visobert_cons`  | + FGM ε=1.0 + consistency 0.5 | 0.7186 |

`TRAIN` **tự phát hiện**: nếu `oof/` và `ckpt/` của vòng thi có mặt thì nạp lại;
nếu không (ví dụ khi BTC chạy trên máy sạch chỉ có `data/`) thì tự huấn luyện
lại toàn bộ từ đầu — khoảng **90 phút trên 1×H100**. Notebook do đó chạy độc lập,
không cần artifact kèm theo.

In [8]:
# Tự phát hiện: nếu thiếu artifact của vòng thi thì huấn luyện lại từ đầu.
# Nhờ vậy notebook chạy được độc lập chỉ với thư mục data/ (cần mạng để tải encoder).
CONFIGS_KEYS = ["visobert_clean", "visobert_fgm", "visobert_cons"]
_have = all(os.path.exists(f"oof/{t}_oof.npz") for t in CONFIGS_KEYS) and \
        all(os.path.exists(f"ckpt/{t}/fold4.pt") for t in CONFIGS_KEYS)
TRAIN = not _have
print("artifact vòng thi:", "CÓ -> nạp lại" if _have else "THIẾU -> huấn luyện từ đầu (~90 phút/1×H100)")
print("TRAIN =", TRAIN)
CONFIGS = {
    "visobert_clean": dict(fgm_eps=0.0, cons=0.0, epochs=10),
    "visobert_fgm":   dict(fgm_eps=1.0, cons=0.0, epochs=8),
    "visobert_cons":  dict(fgm_eps=1.0, cons=0.5, epochs=8),
}
te_pub = pd.read_csv(f"{DATA}/public_test.csv"); te_pub["text"] = te_pub.text.fillna("").astype(str)

if TRAIN:
    tok = AutoTokenizer.from_pretrained(MODEL)
    for tag, cfg in CONFIGS.items():
        oh = np.zeros((len(df),3)); on = np.zeros((len(df),7)); seen = np.zeros(len(df), bool); TH,TN = [],[]
        for k in range(5):
            (s,(vh,vn),(th,tn)), idx = run_fold(df, k, tok, list(te_pub.text), tag=tag, **cfg)
            oh[idx],on[idx],seen[idx] = vh,vn,True; TH.append(th); TN.append(tn)
            print(f"[fold {k}] best SCORE={s:.4f}")
        np.savez(f"oof/{tag}_oof.npz", h=oh, n=on, seen=seen, te_h=np.mean(TH,0), te_n=np.mean(TN,0))
else:
    for tag in CONFIGS:
        print(f"=== {tag} — nhật ký huấn luyện thật trong vòng thi ===")
        log = f"logs/{tag}.log"
        if os.path.exists(log):
            for ln in open(log):
                if re.search(r"fold \d\]|OOF over", ln): print("   ", ln.rstrip())
        print()

artifact vòng thi: CÓ -> nạp lại
TRAIN = False
=== visobert_clean — nhật ký huấn luyện thật trong vòng thi ===
    [fold 0] best SCORE=0.7114
    [fold 1] best SCORE=0.7029
    [fold 2] best SCORE=0.7033
    [fold 3] best SCORE=0.6992
    [fold 4] best SCORE=0.6906
    [visobert_clean] OOF over 51663 rows: hate=0.6972 noise=0.7280 SCORE=0.7018

=== visobert_fgm — nhật ký huấn luyện thật trong vòng thi ===
    [fold 0] best SCORE=0.7277
    [fold 1] best SCORE=0.7169
    [fold 2] best SCORE=0.7132
    [fold 3] best SCORE=0.7126
    [fold 4] best SCORE=0.6961
    [visobert_fgm] OOF over 51663 rows: hate=0.7089 noise=0.7421 SCORE=0.7139

=== visobert_cons — nhật ký huấn luyện thật trong vòng thi ===
    [fold 0] best SCORE=0.7320
    [fold 1] best SCORE=0.7246
    [fold 2] best SCORE=0.7107
    [fold 3] best SCORE=0.7186
    [fold 4] best SCORE=0.7067
    [visobert_cons] OOF over 51663 rows: hate=0.7125 noise=0.7531 SCORE=0.7186



## 5. Đánh giá OOF (51.663 dòng — độ phân giải cao gấp ~4 lần bảng public 3.340 dòng)

In [9]:
def official(yh,ph,yn,pn):
    fh = f1_score(yh,ph,average="macro",labels=range(3),zero_division=0)
    fn = f1_score(yn,pn,average="macro",labels=range(7),zero_division=0)
    return 0.85*fh+0.15*fn, fh, fn

OOF = {t: np.load(f"oof/{t}_oof.npz") for t in CONFIGS}
rows=[]
for t,d in OOF.items():
    m=d["seen"]; s,fh,fn = official(df.y_hate.values[m], d["h"][m].argmax(1), df.y_noise.values[m], d["n"][m].argmax(1))
    rows.append(dict(model=t, hate=round(fh,4), noise=round(fn,4), SCORE=round(s,4)))
print(pd.DataFrame(rows).to_string(index=False))

         model   hate  noise  SCORE
visobert_clean 0.6972 0.7280 0.7018
  visobert_fgm 0.7089 0.7421 0.7139
 visobert_cons 0.7125 0.7531 0.7186


In [10]:
d = OOF["visobert_cons"]; m = d["seen"]
print("=== hate, từng lớp (mô hình tốt nhất) ===")
print(classification_report(df.y_hate.values[m], d["h"][m].argmax(1), target_names=HATE_LABELS, digits=4, zero_division=0))
cm = confusion_matrix(df.y_hate.values[m], d["h"][m].argmax(1), labels=range(3))
print(pd.DataFrame(cm, index=HATE_LABELS, columns=HATE_LABELS).to_string())
print()
for i,nm in enumerate(HATE_LABELS):
    r=cm[i]; print(f"  thực tế {nm:9s} n={r.sum():5d} -> " + ", ".join(f"{HATE_LABELS[j]} {r[j]/r.sum()*100:5.1f}%" for j in range(3)))
print("\n=> Lỗi chủ đạo là ĐỘC HẠI -> CLEAN, không phải ranh giới OFFENSIVE<->HATE.")

=== hate, từng lớp (mô hình tốt nhất) ===
              precision    recall  f1-score   support

       CLEAN     0.9417    0.9465    0.9441     42537
   OFFENSIVE     0.5271    0.5083    0.5175      3555
        HATE     0.6814    0.6703    0.6758      5571

    accuracy                         0.8866     51663
   macro avg     0.7167    0.7084    0.7125     51663
weighted avg     0.8851    0.8866    0.8858     51663

           CLEAN  OFFENSIVE  HATE
CLEAN      40262       1147  1128
OFFENSIVE   1130       1807   618
HATE        1363        474  3734

  thực tế CLEAN     n=42537 -> CLEAN  94.7%, OFFENSIVE   2.7%, HATE   2.7%
  thực tế OFFENSIVE n= 3555 -> CLEAN  31.8%, OFFENSIVE  50.8%, HATE  17.4%
  thực tế HATE      n= 5571 -> CLEAN  24.5%, OFFENSIVE   8.5%, HATE  67.0%

=> Lỗi chủ đạo là ĐỘC HẠI -> CLEAN, không phải ranh giới OFFENSIVE<->HATE.


## 6. Sai số của thước đo — bootstrap

Trước khi tin bất kỳ so sánh nào trên bảng public, phải biết bảng public sai số bao nhiêu.

In [11]:
yh, yn = df.y_hate.values[m], df.y_noise.values[m]
ph, pn = d["h"][m].argmax(1), d["n"][m].argmax(1)
rng = np.random.RandomState(0); sc=[]
for _ in range(400):
    i = rng.choice(len(yh), 3340, replace=True)
    sc.append(official(yh[i],ph[i],yn[i],pn[i])[0])
sc = np.array(sc)
print(f"mô phỏng một lần chấm trên 3.340 mẫu (400 lần bootstrap):")
print(f"  trung bình = {sc.mean():.4f}")
print(f"  SD         = {sc.std():.4f}   <-- sai số chuẩn của MỘT điểm public/private")
print(f"  khoảng 95% = [{np.percentile(sc,2.5):.4f}, {np.percentile(sc,97.5):.4f}]")
print(f"\nOOF trên {len(yh):,} dòng: {official(yh,ph,yn,pn)[0]:.4f}  (sai số ≈ ±{sc.std()/np.sqrt(len(yh)/3340):.4f})")
print("\n=> Chênh lệch dưới ~0.023 trên bảng public KHÔNG phân biệt được.")
print("   Mọi quyết định chọn mô hình đều dựa trên OOF, không dựa trên public.")

mô phỏng một lần chấm trên 3.340 mẫu (400 lần bootstrap):
  trung bình = 0.7188
  SD         = 0.0108   <-- sai số chuẩn của MỘT điểm public/private
  khoảng 95% = [0.6977, 0.7393]

OOF trên 51,663 dòng: 0.7186  (sai số ≈ ±0.0027)

=> Chênh lệch dưới ~0.023 trên bảng public KHÔNG phân biệt được.
   Mọi quyết định chọn mô hình đều dựa trên OOF, không dựa trên public.


## 7. Ensemble — trọng số tối ưu trên OOF, có kiểm soát overfit

In [12]:
import itertools
def search_w(Ps, y, ncls, steps=11):
    best, bw = -1, None
    for combo in itertools.product(range(steps+1), repeat=len(Ps)-1):
        if sum(combo) > steps: continue
        w = np.array(list(combo)+[steps-sum(combo)], float)/steps
        s = f1_score(y, sum(wi*p for wi,p in zip(w,Ps)).argmax(1), average="macro", labels=range(ncls), zero_division=0)
        if s > best: best, bw = s, w
    return bw, best

TAGS = ["visobert_fgm","visobert_cons","visobert_clean"]
mm = OOF[TAGS[0]]["seen"].copy()
for t in TAGS[1:]: mm &= OOF[t]["seen"]
Hs=[OOF[t]["h"][mm] for t in TAGS]; Ns=[OOF[t]["n"][mm] for t in TAGS]
yh2, yn2 = df.y_hate.values[mm], df.y_noise.values[mm]

idx = np.random.RandomState(0).permutation(mm.sum()); A,B = idx[::2], idx[1::2]
hold=[]
for fit,ev in ((A,B),(B,A)):
    w,_ = search_w([P[fit] for P in Hs], yh2[fit], 3)
    hold.append(f1_score(yh2[ev], sum(wi*p[ev] for wi,p in zip(w,Hs)).argmax(1), average="macro", labels=range(3), zero_division=0))
wh_, sh = search_w(Hs, yh2, 3); wn_, sn = search_w(Ns, yn2, 7)
print(f"hate : single tốt nhất={max(f1_score(yh2,P.argmax(1),average='macro',labels=range(3),zero_division=0) for P in Hs):.4f}"
      f"  ensemble held-out={np.mean(hold):.4f}   trọng số={np.round(wh_,3)}")
print(f"noise: single tốt nhất={max(f1_score(yn2,P.argmax(1),average='macro',labels=range(7),zero_division=0) for P in Ns):.4f}"
      f"  ensemble full-fit={sn:.4f}   trọng số={np.round(wn_,3)}")

hate : single tốt nhất=0.7125  ensemble held-out=0.7143   trọng số=[0.182 0.636 0.182]
noise: single tốt nhất=0.7531  ensemble full-fit=0.7579   trọng số=[0.455 0.455 0.091]


## 8. Hiệu chỉnh prior của đầu nhiễu

Tập test có phân bố nhiễu **khác** tập train. Ước lượng bằng bốn thống kê bề mặt
tất định, đều được hiệu chuẩn trên train nơi có nhãn thật.

In [13]:
VN = set("àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ".upper()
         + "àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ")
def sig(t):
    t=str(t)
    return (sum(c in VN for c in t), len(re.findall(r"(\w)\1{2,}",t)),
            len(re.findall(r"[A-Za-zÀ-ỹ][*._][A-Za-zÀ-ỹ]",t)), t.count("  "), len(re.findall(r"[A-Za-zÀ-ỹ]",t)))
cols=["diac","rep3","obf","dsp","alpha"]
te_pri = pd.read_csv(f"{DATA}/private_test.csv"); te_pri["text"]=te_pri.text.fillna("").astype(str)
S = pd.DataFrame([sig(t) for t in te_pri.text], columns=cols)
obs = dict(nodiac=((S.diac==0)&(S.alpha>3)).mean(), obf=(S.obf>0).mean(), rep3=(S.rep3>0).mean(), dsp=(S.dsp>0).mean())
coef = dict(nodiac=(1.375,0.060), obf=(1.153,0.018), rep3=(0.976,0.031), dsp=(0.851,0.013))
qs = {k: (obs[k]-c[1])/c[0] for k,c in coef.items()}
q = float(np.mean(list(qs.values())))
print("tỉ lệ tín hiệu trên private test:", {k: round(v,4) for k,v in obs.items()})
print("q suy ra từ từng phương trình :", {k: round(v,4) for k,v in qs.items()})
print(f"\n=> mỗi loại nhiễu ≈ {q*100:.1f}%,  ORIGINAL ≈ {(1-6*q)*100:.1f}%   (train: ORIGINAL 50%)")
TEST_PRIOR = np.array([1-6*q] + [q]*6); TEST_PRIOR = np.clip(TEST_PRIOR,1e-3,None); TEST_PRIOR /= TEST_PRIOR.sum()

tỉ lệ tín hiệu trên private test: {'nodiac': 0.2446, 'obf': 0.1772, 'rep3': 0.1617, 'dsp': 0.1153}
q suy ra từ từng phương trình : {'nodiac': 0.1343, 'obf': 0.1381, 'rep3': 0.1339, 'dsp': 0.1202}

=> mỗi loại nhiễu ≈ 13.2%,  ORIGINAL ≈ 21.0%   (train: ORIGINAL 50%)


In [14]:
Pn = sum(wi*p for wi,p in zip(wn_, Ns))
train_p = np.bincount(yn2, minlength=7)/len(yn2)
sw = (TEST_PRIOR/train_p)[yn2]
wf1 = lambda yp: f1_score(yn2, yp, average="macro", labels=range(7), zero_division=0, sample_weight=sw)
L = np.log(np.clip(Pn,1e-9,1)); bias_n = np.zeros(7); best = wf1(L.argmax(1))
for _ in range(6):
    imp=False
    for c in range(7):
        bs,bb = best, bias_n[c]
        for g in np.linspace(-2,2,41):
            bias_n[c]=g; s=wf1((L+bias_n).argmax(1))
            if s>bs: bs,bb=s,g
        bias_n[c]=bb
        if bs>best+1e-9: best,imp=bs,True
    if not imp: break
print(f"noise macro-F1 theo prior TEST: argmax={wf1(L.argmax(1)):.4f} -> sau hiệu chỉnh={best:.4f}")
print("bias:", np.round(bias_n,2))
print("\nLƯU Ý: hiệu chỉnh phải theo prior TEST. Tune theo prior TRAIN đẩy ORIGINAL")
print("lên ~34% (thật ~21%) và đã làm mất 0.005 điểm ở một lượt nộp public.")

noise macro-F1 theo prior TEST: argmax=0.7563 -> sau hiệu chỉnh=0.7723
bias: [-0.7 -0.6  0.1 -0.7 -0.4 -0.7 -0.7]

LƯU Ý: hiệu chỉnh phải theo prior TEST. Tune theo prior TRAIN đẩy ORIGINAL
lên ~34% (thật ~21%) và đã làm mất 0.005 điểm ở một lượt nộp public.


## 9. Suy luận trên Private Test và xuất bài nộp

In [15]:
def predict(tag, texts, maxlen=96, bs=256):
    tok = AutoTokenizer.from_pretrained(MODEL)
    dl = DataLoader(DS(list(texts), tok, maxlen), batch_size=bs,
                    collate_fn=lambda b: collate(b, tok.pad_token_id), num_workers=4)
    H,N_ = [],[]
    for f in sorted([f"ckpt/{tag}/{x}" for x in os.listdir(f"ckpt/{tag}") if x.endswith(".pt")]):
        st = torch.load(f, map_location="cpu", weights_only=False)
        mdl = MTL(MODEL).cuda(); mdl.load_state_dict(st["model"]); mdl.eval()
        h,nn_ = [],[]
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
            for b in dl:
                lh,ln = mdl(b["input_ids"].cuda(), b["attention_mask"].cuda())
                h.append(lh.float().softmax(-1).cpu()); nn_.append(ln.float().softmax(-1).cpu())
        H.append(torch.cat(h).numpy()); N_.append(torch.cat(nn_).numpy())
        del mdl; torch.cuda.empty_cache()
    return np.mean(H,0), np.mean(N_,0)

P = {t: predict(t, te_pri.text) for t in TAGS}
# Bài nộp đạt điểm cao nhất (Private 0.720) dùng CHUNG bộ trọng số wn_ cho cả hai
# đầu ra. Trọng số riêng từng task (wh_ cho hate) là biến thể pE_pertask, cũng
# nộp và cho kết quả tương đương — ở đây tái tạo đúng bài 0.720.
W = wn_
te_h = sum(wi*P[t][0] for wi,t in zip(W, TAGS))
te_n = sum(wi*P[t][1] for wi,t in zip(W, TAGS))
te_n = np.log(np.clip(te_n,1e-9,1)) + bias_n

sub = pd.DataFrame({
    "id": te_pri["id"],                                   # CHỈ để ghép dòng, không phải đặc trưng
    "pred_label":      [HATE_LABELS[i]  for i in te_h.argmax(1)],
    "pred_noise_type": [NOISE_LABELS[i] for i in te_n.argmax(1)],
})
sub.to_csv("task1_private_output.csv", index=False, encoding="utf-8")
print("đã ghi task1_private_output.csv  rows =", len(sub))
print()
print(sub.pred_label.value_counts().to_string()); print()
print(sub.pred_noise_type.value_counts().to_string())
print(f"\nkiểm tra: ORIGINAL dự đoán {(sub.pred_noise_type=='ORIGINAL').mean()*100:.1f}%  (ước lượng {(1-6*q)*100:.1f}%)")
print("ID:", "khớp private_test" if set(sub.id)==set(te_pri.id) else "LỆCH", "| trùng lặp:", sub.id.duplicated().sum())

Some weights of XLMRobertaModel were not initialized from the model checkpoint at 5CD-AI/visobert-14gb-corpus and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Some weights of XLMRobertaModel were not initialized from the model checkpoint at 5CD-AI/visobert-14gb-corpus and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Some weights of XLMRobertaModel were not initialized from the model checkpoint at 5CD-AI/visobert-14gb-corpus and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Some weights of XLMRobertaModel were not initialized from the model checkpoint at 5CD-AI/visobert-14gb-corpus and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Some weights of XLMRobertaModel were not initialized from the model checkpoint at 5CD-AI/visobert-14gb-corpus and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Some weights of XLMRobertaModel were not initialized from the model checkpoint at 5CD-AI/visobert-14gb-corpus and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Some weights of XLMRobertaModel were not initialized from the model checkpoint at 5CD-AI/visobert-14gb-corpus and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Some weights of XLMRobertaModel were not initialized from the model checkpoint at 5CD-AI/visobert-14gb-corpus and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Some weights of XLMRobertaModel were not initialized from the model checkpoint at 5CD-AI/visobert-14gb-corpus and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


đã ghi task1_private_output.csv  rows = 3340

pred_label
CLEAN        2789
HATE          343
OFFENSIVE     208

pred_noise_type
ORIGINAL         776
NO_DIACRITICS    472
CHAR_REPEAT      456
OBFUSCATION      449
PUNCT_NOISE      443
TEENCODE         385
MIXED            359

kiểm tra: ORIGINAL dự đoán 23.2%  (ước lượng 21.0%)
ID: khớp private_test | trùng lặp: 0
